In [0]:
# Databricks notebook source
import mlflow
import mlflow.sklearn
from pyspark.sql.functions import *
import pandas as pd
import numpy as np
import logging

# Silenciar warnings do MLflow relacionados ao sandbox do Serverless
logging.getLogger("mlflow").setLevel(logging.ERROR)

print("=" * 60)
print("FASE 5: MACHINE LEARNING")
print("=" * 60)

VOLUME_PATH = "/Volumes/workspace/default/nyc_taxi"
PROCESSED_PATH = f"{VOLUME_PATH}/processed"

# Setup MLflow
experiment_name = "/Shared/nyc_taxi_fare_prediction"
mlflow.set_experiment(experiment_name)
print(f"✅ MLflow experiment: {experiment_name}")

# Carregar dados featured (Fase 3)
df = spark.read.format("delta").load(f"{PROCESSED_PATH}/featured/taxi_featured")
print(f"\n📊 Linhas disponíveis: {df.count():,}")

In [0]:
# TAREFA 5.2: Split Temporal
print("\n" + "="*60)
print("TAREFA 5.2: Split Temporal (Treino/Validação/Teste)")
print("="*60)

# Criar coluna auxiliar de timestamp numérico
df_ts = df.withColumn("pickup_date_unix", unix_timestamp(col("pickup_date")))

# Encontrar os cortes de data que dividem os dados em 70/15/15
cortes = df_ts.approxQuantile("pickup_date_unix", [0.70, 0.85], 0.001)
corte_treino_val, corte_val_teste = cortes[0], cortes[1]

df_train = df_ts.filter(col("pickup_date_unix") <= corte_treino_val)
df_val = df_ts.filter((col("pickup_date_unix") > corte_treino_val) & 
                       (col("pickup_date_unix") <= corte_val_teste))
df_test = df_ts.filter(col("pickup_date_unix") > corte_val_teste)

print(f"📊 Treino:     {df_train.count():,} linhas")
print(f"📊 Validação:  {df_val.count():,} linhas")
print(f"📊 Teste:      {df_test.count():,} linhas")

# Sanity check: confirmar que não há sobreposição temporal
print(f"\n📅 Data máxima do treino:    {df_train.agg(max('pickup_date')).collect()[0][0]}")
print(f"📅 Data mínima da validação: {df_val.agg(min('pickup_date')).collect()[0][0]}")
print(f"📅 Data máxima da validação: {df_val.agg(max('pickup_date')).collect()[0][0]}")
print(f"📅 Data mínima do teste:     {df_test.agg(min('pickup_date')).collect()[0][0]}")

In [0]:
# TAREFA 5.2b: Amostragem (necessário para caber na memória do driver)
print("\n" + "="*60)
print("TAREFA 5.2b: Amostragem para Treino em Scikit-learn")
print("="*60)

FRACAO_AMOSTRA = 0.05  # ~5% → ajuste conforme a memória disponível

train_pd = df_train.sample(withReplacement=False, fraction=FRACAO_AMOSTRA, seed=42).toPandas()
val_pd = df_val.sample(withReplacement=False, fraction=FRACAO_AMOSTRA, seed=42).toPandas()
test_pd = df_test.sample(withReplacement=False, fraction=FRACAO_AMOSTRA, seed=42).toPandas()

print(f"📊 Treino (amostra):    {len(train_pd):,} linhas")
print(f"📊 Validação (amostra): {len(val_pd):,} linhas")
print(f"📊 Teste (amostra):     {len(test_pd):,} linhas")

In [0]:
# TAREFA 5.3: Seleção de Features
print("\n" + "="*60)
print("TAREFA 5.3: Seleção de Features")
print("="*60)

FEATURE_COLS = [
    "trip_distance_km", "pickup_hour", "pickup_day_of_week",
    "is_weekend", "is_rush_hour", "is_airport_trip",
    "passenger_count", "payment_type"
]
TARGET_COL = "fare_amount"

# Remover linhas com nulos nas features/target escolhidas
train_clean = train_pd[FEATURE_COLS + [TARGET_COL]].dropna()
val_clean = val_pd[FEATURE_COLS + [TARGET_COL]].dropna()
test_clean = test_pd[FEATURE_COLS + [TARGET_COL]].dropna()

X_train, y_train = train_clean[FEATURE_COLS], train_clean[TARGET_COL]
X_val, y_val = val_clean[FEATURE_COLS], val_clean[TARGET_COL]
X_test, y_test = test_clean[FEATURE_COLS], test_clean[TARGET_COL]

# Converter colunas inteiras para float64 (evita warning de schema no MLflow)
X_train = X_train.astype("float64")
X_val = X_val.astype("float64")
X_test = X_test.astype("float64")

print(f"✅ Features selecionadas: {FEATURE_COLS}")
print(f"✅ Target: {TARGET_COL}")
print(f"\n📊 Treino: {len(X_train):,} | Validação: {len(X_val):,} | Teste: {len(X_test):,}")

print(f"\n📊 Correlação com target:")
print(train_clean.corr()[TARGET_COL].sort_values(ascending=False))

In [0]:
# TAREFA 5.5: Linear Regression (Baseline)
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("\n" + "="*60)
print("TAREFA 5.5: Treinando Linear Regression")
print("="*60)

with mlflow.start_run(run_name="linear_regression_baseline"):
    lr = LinearRegression()
    lr.fit(X_train, y_train)
    
    y_pred = lr.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("features", FEATURE_COLS)
    mlflow.log_param("train_size", len(X_train))
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2_score", r2)
    mlflow.sklearn.log_model(lr, name="model", input_example=X_train.head())
    
    print(f"✅ RMSE: ${rmse:.2f} | MAE: ${mae:.2f} | R²: {r2:.4f}")

In [0]:
# TAREFA 5.5b: Random Forest
from sklearn.ensemble import RandomForestRegressor

print("\n" + "="*60)
print("TAREFA 5.5b: Treinando Random Forest")
print("="*60)

with mlflow.start_run(run_name="random_forest"):
    rf = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    
    y_pred = rf.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    mlflow.log_param("model_type", "RandomForestRegressor")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 12)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2_score", r2)
    mlflow.sklearn.log_model(rf, name="model", input_example=X_train.head())
    
    print(f"✅ RMSE: ${rmse:.2f} | MAE: ${mae:.2f} | R²: {r2:.4f}")

In [0]:
# TAREFA 5.5c: Gradient Boosting
from sklearn.ensemble import GradientBoostingRegressor

print("\n" + "="*60)
print("TAREFA 5.5c: Treinando Gradient Boosting")
print("="*60)

with mlflow.start_run(run_name="gradient_boosting"):
    gb = GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42)
    gb.fit(X_train, y_train)
    
    y_pred = gb.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    mlflow.log_param("model_type", "GradientBoostingRegressor")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 5)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2_score", r2)
    mlflow.sklearn.log_model(gb, name="model", input_example=X_train.head())
    
    print(f"✅ RMSE: ${rmse:.2f} | MAE: ${mae:.2f} | R²: {r2:.4f}")

In [0]:
# TAREFA 5.8: Comparação de Modelos (via MLflow)
print("\n" + "="*60)
print("TAREFA 5.8: Comparação de Modelos")
print("="*60)

runs = mlflow.search_runs(experiment_names=[experiment_name], order_by=["metrics.rmse ASC"])
display(runs[["tags.mlflow.runName", "metrics.rmse", "metrics.mae", "metrics.r2_score"]])

melhor_run = runs.iloc[0]
print(f"\n🏆 Melhor modelo: {melhor_run['tags.mlflow.runName']}")
print(f"   RMSE: ${melhor_run['metrics.rmse']:.2f}")
print(f"   R²:   {melhor_run['metrics.r2_score']:.4f}")

In [0]:
# TAREFA 5.9: Feature Importance
print("\n" + "="*60)
print("TAREFA 5.9: Feature Importance")
print("="*60)

# Ajuste conforme o melhor modelo real (rf ou gb) apontado na Célula 8
modelo_vencedor = gb if "gradient" in melhor_run["tags.mlflow.runName"] else rf

importancias = pd.DataFrame({
    "feature": FEATURE_COLS,
    "importance": modelo_vencedor.feature_importances_
}).sort_values("importance", ascending=False)

display(importancias)

In [0]:
# TAREFA 5.10: Model Registry com Registro Condicional
print("\n" + "="*60)
print("TAREFA 5.10: Registro Condicional do Melhor Modelo")
print("="*60)

from mlflow import MlflowClient
from mlflow.exceptions import MlflowException

client = MlflowClient()
model_name = "workspace.default.nyc_taxi_fare_prediction"
alias = "champion"

melhor_run_id = melhor_run["run_id"]
novo_rmse = melhor_run["metrics.rmse"]

# 1. Tentar buscar o champion atual (pode não existir na primeira vez)
rmse_champion_atual = None
try:
    versao_champion = client.get_model_version_by_alias(model_name, alias)
    run_champion = client.get_run(versao_champion.run_id)
    rmse_champion_atual = run_champion.data.metrics.get("rmse")
    print(f"📊 Champion atual: versão {versao_champion.version}, RMSE ${rmse_champion_atual:.2f}")
except MlflowException:
    print("ℹ️  Nenhum champion registrado ainda — este será o primeiro")

print(f"📊 Novo modelo candidato: RMSE ${novo_rmse:.2f}")

# 2. Decidir se registra e atualiza o alias
deve_registrar = (rmse_champion_atual is None) or (novo_rmse < rmse_champion_atual)

if deve_registrar:
    model_uri = f"runs:/{melhor_run_id}/model"
    nova_versao = mlflow.register_model(model_uri, model_name)
    
    client.set_registered_model_alias(
        name=model_name,
        alias=alias,
        version=nova_versao.version
    )
    
    print(f"\n✅ Novo campeão registrado!")
    print(f"   Versão: {nova_versao.version}")
    print(f"   RMSE: ${novo_rmse:.2f} (anterior: "
          f"${rmse_champion_atual:.2f})" if rmse_champion_atual else "(primeiro registro)")
else:
    print(f"\n⏭️  Modelo NÃO registrado — o champion atual (RMSE ${rmse_champion_atual:.2f}) "
          f"já é melhor ou igual ao novo candidato (RMSE ${novo_rmse:.2f})")
    print(f"   Nenhuma nova versão foi criada")

In [0]:
# Diagnóstico rápido: Overfitting (treino vs teste)
print("=" * 60)
print("DIAGNÓSTICO: Treino vs Teste (checagem de overfitting)")
print("=" * 60)

y_pred_train = gb.predict(X_train)
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
r2_train = r2_score(y_train, y_pred_train)

print(f"Treino → RMSE: ${rmse_train:.2f} | R²: {r2_train:.4f}")
print(f"Teste  → RMSE: ${rmse:.2f} | R²: {r2:.4f}")

diferenca_pct = np.abs(rmse_train - rmse) / rmse_train * 100
print(f"\nDiferença RMSE treino/teste: {diferenca_pct:.1f}%")
if diferenca_pct < 10:
    print("✅ Performance consistente entre treino e teste (sem overfitting aparente)")
else:
    print("⚠️ Diferença significativa - investigar overfitting")

## 📋 Resumo — Fase 5: Machine Learning

### Split Temporal (sem embaralhar no tempo)
- Treino: 33.182.884 linhas (até 2016-02-25)
- Validação: 6.766.255 linhas (2016-02-26 a 2016-03-13)
- Teste: 6.933.011 linhas (a partir de 2016-03-14)
- Amostragem para scikit-learn: 5% de cada partição

### Comparação de Modelos
| Modelo | RMSE | MAE | R² |
|---|---|---|---|
| Linear Regression | $3,23 | $1,69 | 0,9065 |
| Random Forest | $2,73 | $1,33 | 0,9334 |
| **Gradient Boosting** 🏆 | $2,71 | $1,33 | 0,9339 |

### Checagem de Overfitting (Gradient Boosting)
- Treino → RMSE: $2,58 | R²: 0,9349
- Teste → RMSE: $2,71 | R²: 0,9339
- Diferença: 5,2% — sem overfitting aparente

### Feature Importance (Gradient Boosting)
- trip_distance_km: 97,5% — domina completamente, refletindo que a
  tarifa é calculada pelo taxímetro majoritariamente por distância
- is_airport_trip: 1,4%
- Demais features (hora, dia, payment_type, etc.): impacto marginal

### Modelo Registrado
- Nome: workspace.default.nyc_taxi_fare_prediction
- Alias: champion (versão 3, RMSE $2,71)
- Registro condicional: só cria nova versão e move o alias quando o
  RMSE do novo modelo supera o champion atual — evita duplicatas
  desnecessárias no histórico

### Observação
- Alta importância de trip_distance_km sugere que features temporais/
  comportamentais serão mais relevantes em um futuro modelo de
  previsão de GORJETA, não de tarifa

In [0]:
# TAREFA 6.1: Validação Cruzada Temporal
print("\n" + "="*60)
print("TAREFA 6.1: Validação Cruzada Temporal (3 Folds)")
print("="*60)

from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import GradientBoostingRegressor

# Combinar treino + validação (amostra) para fazer CV temporal
cv_pd = pd.concat([train_clean, val_clean]).reset_index(drop=True)
X_cv = cv_pd[FEATURE_COLS].astype("float64")
y_cv = cv_pd[TARGET_COL]

tscv = TimeSeriesSplit(n_splits=3)
resultados_cv = []

for fold, (idx_treino, idx_val) in enumerate(tscv.split(X_cv), 1):
    X_fold_treino, X_fold_val = X_cv.iloc[idx_treino], X_cv.iloc[idx_val]
    y_fold_treino, y_fold_val = y_cv.iloc[idx_treino], y_cv.iloc[idx_val]
    
    modelo_fold = GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42)
    modelo_fold.fit(X_fold_treino, y_fold_treino)
    
    y_pred_fold = modelo_fold.predict(X_fold_val)
    rmse_fold = np.sqrt(mean_squared_error(y_fold_val, y_pred_fold))
    r2_fold = r2_score(y_fold_val, y_pred_fold)
    
    resultados_cv.append({"fold": fold, "rmse": rmse_fold, "r2": r2_fold})
    print(f"Fold {fold}: RMSE=${rmse_fold:.2f} | R²={r2_fold:.4f} "
          f"(treino: {len(idx_treino):,} | val: {len(idx_val):,})")

rmse_medio = np.mean([r["rmse"] for r in resultados_cv])
rmse_std = np.std([r["rmse"] for r in resultados_cv])

print(f"\n📊 RMSE médio: ${rmse_medio:.2f} (±${rmse_std:.2f})")
print(f"✅ Estabilidade: {'boa' if rmse_std < 0.5 else 'moderada'} entre os folds")

In [0]:
# TAREFA 6.2: Análise de Resíduos
print("\n" + "="*60)
print("TAREFA 6.2: Análise de Resíduos")
print("="*60)

y_pred_test = gb.predict(X_test)
residuos = y_test.values - y_pred_test

resid_df = pd.DataFrame({
    "real": y_test.values,
    "previsto": y_pred_test,
    "residuo": residuos,
    "pickup_hour": test_clean["pickup_hour"].values,
    "is_airport_trip": test_clean["is_airport_trip"].values
})

print(f"📊 Estatísticas dos Resíduos:")
print(f"   Média:      ${resid_df['residuo'].mean():.3f} (deve ser próxima de 0)")
print(f"   Desvio:     ${resid_df['residuo'].std():.3f}")
print(f"   Mín/Máx:    ${resid_df['residuo'].min():.2f} / ${resid_df['residuo'].max():.2f}")

# Maiores erros absolutos
print(f"\n🚩 Top 5 maiores erros absolutos:")
top_erros = resid_df.reindex(resid_df["residuo"].abs().sort_values(ascending=False).index).head(5)
display(top_erros)

In [0]:
# TAREFA 6.2b: Resíduos por Segmento
print("\n" + "="*60)
print("TAREFA 6.2b: Erro Médio por Segmento")
print("="*60)

erro_por_aeroporto = resid_df.groupby("is_airport_trip")["residuo"].agg(
    erro_medio="mean", erro_absoluto_medio=lambda x: x.abs().mean(), qtd="count"
)
print("Erro por tipo de viagem (0=Standard, 1=Aeroporto):")
display(erro_por_aeroporto)

erro_por_hora = resid_df.groupby("pickup_hour")["residuo"].agg(
    erro_medio="mean", erro_absoluto_medio=lambda x: x.abs().mean()
).round(2)
print("\nErro médio por hora do dia:")
display(erro_por_hora)

In [0]:
# TAREFA 6.2c: Visualização de Resíduos
import matplotlib.pyplot as plt
import builtins

amostra_plot = resid_df.sample(builtins.min(5000, len(resid_df)), random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(amostra_plot["real"], amostra_plot["previsto"], alpha=0.3, s=10)
axes[0].plot([0, 100], [0, 100], 'r--', label="Previsão perfeita")
axes[0].set_xlabel("Tarifa Real ($)")
axes[0].set_ylabel("Tarifa Prevista ($)")
axes[0].set_title("Real vs Previsto")
axes[0].set_xlim(0, 100)
axes[0].set_ylim(0, 100)
axes[0].legend()

axes[1].hist(resid_df["residuo"], bins=50, edgecolor='black')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel("Resíduo (Real - Previsto)")
axes[1].set_title("Distribuição dos Resíduos")

plt.tight_layout()
plt.show()

## 📋 Resumo — Fase 6: Validação & Avaliação

### Validação Cruzada Temporal (3 Folds)
| Fold | RMSE | R² |
|---|---|---|
| 1 | $2,83 | 0,9226 |
| 2 | $3,04 | 0,9166 |
| 3 | $2,73 | 0,9312 |

- RMSE médio: $2,86 (±$0,13) — próximo do holdout ($2,71), boa estabilidade

### Análise de Resíduos
- Média do resíduo: $0,177 (próxima de 0 — sem viés sistemático geral)
- Desvio padrão: $2,709
- Distribuição concentrada perto de zero, com cauda longa à direita
  (o modelo tende a SUBESTIMAR uma pequena fração de tarifas altas)

### Achado Principal: Tarifas Negociadas (RatecodeID=5) Não Seguem o Padrão
- O gráfico Real vs Previsto revela uma faixa horizontal (~$50-55 previsto)
  para tarifas reais entre $50-95 — o modelo "trava" nesse valor
- Causa provável: viagens com RatecodeID=5 ("Negotiated fare") têm tarifa
  definida por acordo, não pela fórmula distância×taxímetro
- Como RatecodeID não é usado diretamente como feature (só via
  is_airport_trip), o modelo não consegue identificar esses casos

### Erro por Segmento
- Standard: erro absoluto médio $1,31
- Aeroporto: erro absoluto médio $2,16 (esperado, dado o valor
  absoluto maior das tarifas)

### Recomendação para Melhoria Futura
- Incluir RatecodeID como feature categórica (one-hot encoding),
  permitindo ao modelo aprender que tarifas negociadas seguem lógica
  diferente, reduzindo os erros na cauda da distribuição